# Catagotchi (Virtual Pet)

**Authors**: Abhishek Sagar, Jivko Katzarov  
**Semester**: Summer 2026  

## Description
In this project, we implement a desktop-based virtual pet simulator called **Catagotchi** (formerly named `FurWatch`). The game simulates a living pet cat with distinct statistics that decay over time. The user must feed, clean, play with, and put the pet to sleep to maintain its health and happiness.

### Goals of the Project
1. **Real-time Simulation**: Passive decay of four primary stats (Hunger, Happiness, Energy, Hygiene) using a non-blocking asynchronous event loop.
2. **Extensible Architecture**: Separate the GUI engine from additional features and visual configurations (Light/Dark themes) using a dynamic module loading system.
3. **Dynamic Asset Rendering**: Swapping optimized, transparent background PNG images to visually display the pet's current need states.
4. **Smart Need Engine**: Programmatic button-highlighting assistant to guide users based on stats alerts and sleep status.
5. **State Persistence**: Save and load statistics to local JSON files so the pet state persists across application runs.

### Roadmap to Reach the Goal
1. **Core Loop & GUI Setup**: Build the layout and stats decay framework in `main.py` using Tkinter.
2. **Native Preference Styling**: Implement cross-platform native dark/light styling filters in `theme.py`.
3. **Modular Package Design**: Migrate optional components into an isolated, dynamic package in `/features`.
4. **Asset Optimization**: Sourcing, background removal (chroma keying), and scaling of PNG sprites to `128x128` pixels.
5. **State Save & Restart**: Add auto-save capabilities and in-app game restart triggers.

### Responsibilities
We divided the project responsibilities as follows:
- **Jivko Katzarov**: Logic ideation, architecture mapping, flowchart design, character asset design, green screen isolation, and image optimization.
- **Abhishek Sagar**: Project documentation, UI window styling and layout, core game loop engine, dynamic plugin loader, theme detection, active button styling overrides, and JSON auto-save persistence.

## Project Code

### Needed Data Structures
The application relies on primitive variables for statistics and uses standard Python collections for callback systems:
* **`self.on_tick_hooks`**: A list of callback functions called dynamically at the end of each game tick.
* **`self.features`**: A list instantiating all active, dynamically loaded feature classes.
* **JSON State Schema**: Auto-save records are structured in `pet_state.json` as:
  ```json
  {
      "hunger": 100, 
      "happiness": 100, 
      "energy": 100, 
      "hygiene": 100, 
      "is_sleeping": false, 
      "is_alive": true
  }
  ```

### Core Loop & Passive Decay
The simulation relies on a 3-second tick interval using Tkinter's non-blocking `.after()` clock. The code below showcases the core state management of the decay logic.

In [1]:
# Core decay and game tick logic function
def calculate_decay(hunger, happiness, energy, hygiene, is_sleeping):
    if is_sleeping:
        energy = min(100, energy + 10)
        hunger = max(0, hunger - 1)
        hygiene = max(0, hygiene - 1)
    else:
        hunger = max(0, hunger - 4)
        happiness = max(0, happiness - 3)
        energy = max(0, energy - 2)
        hygiene = max(0, hygiene - 3)
    return hunger, happiness, energy, hygiene

We verify this decay logic via unit tests:

In [2]:
# Test cases for passive decay rates
# 1. Test normal awake decay
hu, ha, en, hy = calculate_decay(100, 100, 100, 100, is_sleeping=False)
assert hu == 96 and ha == 97 and en == 98 and hy == 97, "Awake decay calculation is incorrect"

# 2. Test sleeping energy recovery and reduced decay
hu, ha, en, hy = calculate_decay(100, 100, 50, 100, is_sleeping=True)
assert en == 60 and hu == 99 and hy == 99, "Sleeping stats calculations are incorrect"

print("Decay calculations assertions passed!")

Decay calculations assertions passed!


### Smart Recommendations Feature
The recommendations module parses stats and suggests buttons to click. If the pet is asleep and has pending critical needs, the sleep/wake button is highlighted, advising the user to wake it up first.

In [3]:
# Recommendation logic filter function
def get_recommendations(hunger, happiness, energy, hygiene, is_sleeping):
    other_stats_low = (hunger < 30 or happiness < 30 or hygiene < 30)
    recommend_wake_up = is_sleeping and (energy >= 100 or other_stats_low)

    return {
        "feed": hunger < 30 and not is_sleeping,
        "play": happiness < 30 and not is_sleeping,
        "sleep": (energy < 30 and not is_sleeping) or recommend_wake_up,
        "clean": hygiene < 30 and not is_sleeping
    }

We run assertions to check the recommendations engine:

In [4]:
# Test cases for recommendations engine
# 1. Test awake state hungry recommendation
recs = get_recommendations(hunger=20, happiness=100, energy=100, hygiene=100, is_sleeping=False)
assert recs["feed"] == True and recs["sleep"] == False, "Failed to recommend food when hungry and awake"

# 2. Test sleep state preservation (should not recommend food while asleep, but recommend waking up)
recs = get_recommendations(hunger=20, happiness=100, energy=50, hygiene=100, is_sleeping=True)
assert recs["feed"] == False and recs["sleep"] == True, "Failed to prioritize waking up when hungry and asleep"

print("Recommendations engine assertions passed!")

Recommendations engine assertions passed!


### State Persistence (JSON Save/Load)
This module handles writing statistics to `pet_state.json` and loading them back on initialization.

In [5]:
import json
import os

def save_state(filepath, hunger, happiness, energy, hygiene, is_sleeping, is_alive):
    state = {
        "hunger": hunger,
        "happiness": happiness,
        "energy": energy,
        "hygiene": hygiene,
        "is_sleeping": is_sleeping,
        "is_alive": is_alive
    }
    with open(filepath, "w") as f:
        json.dump(state, f)

def load_state(filepath):
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            return json.load(f)
    return None

We test state serialization and file reads:

In [6]:
# Test cases for persistence module
test_file = "test_pet_state.json"
save_state(test_file, 80, 70, 60, 50, is_sleeping=False, is_alive=True)

loaded = load_state(test_file)
assert loaded is not None, "Failed to load state file"
assert loaded["hunger"] == 80 and loaded["energy"] == 60 and loaded["is_alive"] == True, "Loaded data mismatch"

# Clean up test file
if os.path.exists(test_file):
    os.remove(test_file)

print("Persistence module assertions passed!")

Persistence module assertions passed!


### Main Program Flowchart

Here is the execution flowchart of the main application loop:

![Flowchart of the main loop](flowchart.png)

### Running the Main Game
You can launch the GUI of the application by running the following cell. (Note: A graphical environment display server is required to launch Tkinter windows).

In [7]:
# Function to launch main Tkinter window
def run_game():
    import tkinter as tk
    from main import TamagotchiGame
    
    # Check if graphical display is available
    try:
        root = tk.Tk()
        app = TamagotchiGame(root)
        # Schedule auto-close after 1 second if running headlessly/automated
        # root.after(1000, root.destroy)
        root.mainloop()
        print("GUI closed successfully.")
    except Exception as e:
        print(f"Skipping GUI launch: {e} (Expected in headless or notebook environments)")

run_game()

GUI closed successfully.


## Discussion
### Challenges Faced
* **macOS Rendering Style Locks**: Override configurations had to be engineered targeting `activeforeground` and `highlightbackground` configurations since standard native background colors are ignored on macOS.
* **Dynamic Styling Race Conditions**: Loading the theme styles prior to construction of modular features caused background disparities in dynamically added panels. Resolving this required setting `current_applied_theme = None` inside the features initialization to force a redraw update.
* **Tkinter event-loop lockups**: Managing background tasks using standard `threading.Thread` created GUI rendering warnings. Using standard native `.after()` scheduling removed rendering overheads.

### Further Improvements
The code structure can be further optimized and expanded:
* **Asset Animation**: Replacing static 1:1 PNGs with multi-frame loop animations by parsing GIFs using Pillow arrays.
* **Audio Modules**: Integrating cross-platform audio players (such as `simpleaudio` or `playsound`) to trigger state noises.
* **Advanced State Save**: Adapting the local JSON saver into a persistent database like SQLite to record a complete historic timeline of the pet's lifetime metrics.